# RAPTOR Tree Explorer

Query the RAPTOR hierarchical tree structure stored in Chroma DB.

In [2]:
import chromadb
# from chromadb.utils.embedding_functions import DefaultEmbeddingFunction
from pathlib import Path

# Use absolute path to backend chromadb
backend_dir = Path('/Users/qtpie/repos/FinRAG/backend')
chroma_path = str(backend_dir / 'chromadb')

# Connect to the backend Chroma database
client = chromadb.PersistentClient(path=chroma_path)
print('✓ Chroma path:', chroma_path)
print('✓ Available collections:', client.list_collections())

AttributeError: module 'chromadb' has no attribute 'PersistentClient'

In [3]:
# Load the RAPTOR collection
col = client.get_collection('finfrag_raptor')
print(f'✓ Collection: {col.name}')
print(f'✓ Total documents: {col.count()}')

NameError: name 'client' is not defined

In [4]:
# Fetch root nodes (parent_id = '__raptor_root__')
root_result = col.get(
    where={'parent_id': '__raptor_root__'},
    include=['documents', 'metadatas'],
    limit=20
)

print(f'✓ Root nodes: {len(root_result["ids"])}\n')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
for i, (doc_id, doc, meta) in enumerate(zip(root_result['ids'], root_result['documents'], root_result['metadatas']), 1):
    print(f'\n{i}. ID: {doc_id}')
    print(f'   Level: {meta.get("level")}, Summary: {meta.get("is_summary")}, Parent: {meta.get("parent_id")}')
    print(f'   Content: {doc[:280].replace(chr(10), " ")}...')

NameError: name 'col' is not defined

In [ ]:
# Inspect children of the first root node
if root_result['ids']:
    first_root_id = root_result['ids'][0]
    print(f'Fetching children of root node:\n{first_root_id}\n')
    print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
    
    children = col.get(
        where={'parent_id': first_root_id},
        include=['documents', 'metadatas'],
        limit=20
    )
    
    print(f'\n✓ Children count: {len(children["ids"])}\n')
    for i, (doc_id, doc, meta) in enumerate(zip(children['ids'], children['documents'], children['metadatas']), 1):
        print(f'{i}. ID: {doc_id}')
        print(f'   Level: {meta.get("level")}, Summary: {meta.get("is_summary")}')
        print(f'   Content: {doc[:250].replace(chr(10), " ")}...\n')

In [ ]:
# Tree Statistics
all_docs = col.get(include=['metadatas'], limit=5000)
metadatas = all_docs['metadatas']

from collections import Counter

levels = Counter(m.get('level') for m in metadatas)
summary_types = Counter(m.get('is_summary', False) for m in metadatas)

print('━━━ RAPTOR Tree Statistics ━━━')
print(f'Total nodes: {len(all_docs["ids"])}')
print(f'Levels: {dict(sorted(levels.items()))}')
print(f'Node types - Summary: {summary_types[True]}, Leaf: {summary_types[False]}')